In [1]:
import os
import pandas as pd
import numpy as np

from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, make_scorer, f1_score
from sklearn.utils.class_weight import compute_class_weight

In [2]:
# load final model data
df = pd.read_csv("model_data_final_v2.csv")

# multiclass mapping of target varible: InjurySeverity
df['InjurySeverity'].value_counts()
df['InjurySeverity'] = df['InjurySeverity'].map({5:0, 4:0, 3:1, 2:1, 1:2})
df['InjurySeverity'].value_counts()

# drop redundant cols
drop_cols = ['IncidentID', 
             'IncidentDateTime',
             'has_child',
             'first_harmful_fixed_object',
             'is_age_middle']
df.drop(drop_cols, axis = 1, inplace = True)

In [3]:
# features and target variable
X = df.drop('InjurySeverity', axis = 1).copy()
y = df['InjurySeverity'].copy()

# train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify = y, test_size = 0.3, random_state = 42)

In [4]:
# compute class weights and map to sample weights
classes = np.unique(y_train)
class_weights = compute_class_weight(class_weight = 'balanced', classes = classes, y = y_train)
weight_dict = {i: class_weights[i] for i in range(len(class_weights))}
sample_weights = y_train.map(weight_dict)

In [5]:
# define model
xgb = XGBClassifier(
    objective = 'multi:softprob',
    num_class = 3,
    random_state = 42
)

# hyperparameter grid
param_grid = {
    'max_depth': [2, 3, 5],
    'learning_rate': [0.1, 0.2],
    'n_estimators': [50, 100, 200],
    'min_child_weight': [2, 3, 5],
    'gamma': [0.2, 0.5],
    'reg_alpha': [0.1, 0.5],
    'reg_lambda': [0.1, 0.5],
    'subsample': [0.8],
    'colsample_bytree': [0.8]
}

# stratified 5-fold CV
cv = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)

# use macro F1 as scoring metric
scorer = make_scorer(f1_score, average = 'macro')

# grid search with sample weights
grid_search = GridSearchCV(
    estimator = xgb,
    param_grid = param_grid,
    cv = cv,
    scoring = scorer,
    verbose = 1,
    n_jobs = -1
)

grid_search.fit(X_train, y_train, sample_weight = sample_weights)
best_model = grid_search.best_estimator_
print(f"\nBest parameters: {grid_search.best_params_}\n")

# predict on test set
y_pred = best_model.predict(X_test)

Fitting 5 folds for each of 432 candidates, totalling 2160 fits

Best parameters: {'colsample_bytree': 0.8, 'gamma': 0.5, 'learning_rate': 0.2, 'max_depth': 5, 'min_child_weight': 3, 'n_estimators': 100, 'reg_alpha': 0.5, 'reg_lambda': 0.5, 'subsample': 0.8}



In [6]:
# model evaluation
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names = ['KA', 'BC', 'O'])

# confusion matrix
class_names = ['KA', 'BC', 'O']
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm,
    index = [f'Actual_{cls}' for cls in class_names],
    columns = [f'Predicted_{cls}' for cls in class_names]
)

print(f"XGBoost")
print(f"Accuracy: {round(accuracy, 4)}\n")
print(f"Classification Report: \n{report}\n")
print(f"Confusion Matrix: \n{cm_df}\n")

report_dict = classification_report(y_test, y_pred, target_names = ['KA', 'BC', 'O'], output_dict = True)

f1_scores = [
    report_dict['KA']['f1-score'],
    report_dict['BC']['f1-score'],
    report_dict['O']['f1-score']
]

harmonic_mean_f1 = len(f1_scores) / sum(1.0 / f for f in f1_scores if f > 0)
print(f"Harmonic Mean of F1-scores: {round(harmonic_mean_f1, 4)}")

XGBoost
Accuracy: 0.5842

Classification Report: 
              precision    recall  f1-score   support

          KA       0.20      0.29      0.24        34
          BC       0.36      0.43      0.39       148
           O       0.77      0.68      0.72       364

    accuracy                           0.58       546
   macro avg       0.44      0.47      0.45       546
weighted avg       0.62      0.58      0.60       546


Confusion Matrix: 
           Predicted_KA  Predicted_BC  Predicted_O
Actual_KA            10            18            6
Actual_BC            17            63           68
Actual_O             24            94          246

Harmonic Mean of F1-scores: 0.3657
